In [9]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_ollama import ChatOllama
from dotenv import load_dotenv



In [10]:
load_dotenv()
model = ChatOllama(model="hermes3:8b")


In [11]:
class BLOGState(TypedDict):
    topic: str
    outline: str
    blog_content: str


In [12]:
def generate_outline(state: BLOGState) -> BLOGState:
    topic = state["topic"]
    outline_prompt = f"Generate a detailed outline for a blog post about {topic}."
    outline = model.invoke(outline_prompt).content
    state["outline"] = outline
    return state

In [13]:
def generate_blog_content(state: BLOGState) -> BLOGState:
    outline = state["outline"]
    content_prompt = f"Write a blog post based on the following outline:\n{outline}"
    blog_content = model.invoke(content_prompt).content
    state["blog_content"] = blog_content
    return state

In [14]:
graph = StateGraph(BLOGState)

In [15]:
graph.add_node("generate_outline",generate_outline)
graph.add_node("generate_blog_content",generate_blog_content)

graph.add_edge(START,"generate_outline")
graph.add_edge("generate_outline","generate_blog_content")
graph.add_edge("generate_blog_content",END)

workflow = graph.compile()



In [ ]:
initial_state = BLOGState(topic="The benefits of using state graphs for LLM workflows", outline="", blog_content="")

final_state = workflow.invoke(initial_state)

print(final_state["blog_content"])

